<a href="https://colab.research.google.com/github/asherkhan2412-blip/sp500-equity-forecasting/blob/main/Equity_Price_Forecasting.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import yfinance as yf

TICKERS = ['AAPL', 'MSFT', 'JPM', 'GS', 'BAC', 'XOM', 'JNJ', 'AMZN', 'TSLA', 'NVDA']

raw = yf.download(TICKERS, start='2010-01-01', auto_adjust=True, progress=False)
df = raw['Close'].copy()

print("Number of rows is:", df.shape)
print(df.head())
df.info()



In [ ]:
import matplotlib.pyplot as plt

missing_values = df.isnull().sum()
print('Missing values per column:', missing_values)

highest_missing = missing_values.sort_values(ascending=False).head(3)
print('Top 3 columns with the highest missing values:', highest_missing)

lowest_missing = missing_values.sort_values(ascending=True).head(3)
print('Top 3 columns with the lowest missing values:', lowest_missing)

missing_percentage = (missing_values / len(df)) * 100
missing_percentage = missing_percentage.sort_values(ascending=False)

highlight_columns = highest_missing.index.tolist()
bar_colours = []

for column_name in missing_percentage.index:
    if column_name in highlight_columns:
        bar_colours.append('red')
    else:
        bar_colours.append('blue')

plt.figure(figsize=(10, 5))
plt.barh(missing_percentage.index, missing_percentage.values, color=bar_colours)
plt.gca().invert_yaxis()
plt.title("Percentage of Missing Values by Ticker")
plt.xlabel("Missing Values (%)")
plt.ylabel("Tickers")

position = 0
for column_name in missing_percentage.index:
    if column_name in highlight_columns:
        percentage_value = missing_percentage[column_name]
        plt.text(percentage_value + 0.5, position, str(round(percentage_value, 2)) + "%", va='center')
    position = position + 1

plt.show()

# Forward-fill any gaps from staggered listing dates, then drop remaining NaNs
df = df.ffill().dropna()
print(f"\nRows after cleaning: {len(df)}")

In [ ]:
# yfinance already returns a datetime index — sort to confirm chronological order
df = df.sort_index()
df.head()

In [ ]:
import matplotlib.pyplot as plt

ticker = 'MSFT'

plt.figure(figsize=(12, 5))
plt.plot(df.index, df[ticker], linestyle='-', label=ticker)
plt.title(f'Adjusted Close Price Over Time for {ticker}')
plt.xlabel('Date')
plt.ylabel('Price (USD)')
plt.legend()
plt.grid(True)
plt.show()

In [ ]:
ticker = 'MSFT'
window_size = 60  # roughly 3 months of trading days

price_series = df[ticker]
rolling_mean = price_series.rolling(window=window_size).mean()
rolling_std  = price_series.rolling(window=window_size).std()

plt.figure(figsize=(13, 5))
plt.plot(price_series.index, price_series,
         label='Daily Close', color='lightsteelblue', linewidth=0.8, alpha=0.7)
plt.plot(rolling_mean.index, rolling_mean,
         label=f'{window_size}-day Rolling Mean', color='steelblue', linewidth=1.5)
plt.fill_between(rolling_mean.index,
                 rolling_mean - rolling_std,
                 rolling_mean + rolling_std,
                 color='steelblue', alpha=0.15, label='±1 Std Dev')

plt.title(f'{ticker} — Rolling Mean & Std Dev (window = {window_size} days)')
plt.xlabel('Date')
plt.ylabel('Price (USD)')
plt.legend()
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

ticker_colours = {
    'AAPL': 'blue',
    'MSFT': 'orange',
    'JPM':  'green',
    'GS':   'red',
    'BAC':  'purple',
    'XOM':  'brown',
    'JNJ':  'pink',
    'AMZN': 'grey',
    'TSLA': 'cyan',
    'NVDA': 'magenta'
}

def calculate_iqr_bounds(series):
    Q1 = series.quantile(0.25)
    Q3 = series.quantile(0.75)
    IQR = Q3 - Q1
    lower_bound = Q1 - 1.5 * IQR
    upper_bound = Q3 + 1.5 * IQR
    return lower_bound, upper_bound

plt.figure(figsize=(13, 5))

for ticker, colour in ticker_colours.items():

    values = df[ticker].dropna()

    sns.kdeplot(values, label=ticker, color=colour, alpha=0.7)

    lower, upper = calculate_iqr_bounds(values)
    plt.axvline(upper, color=colour, linestyle='--', linewidth=0.8, alpha=0.6)

plt.title('KDE Plot of Equity Close Prices with Upper IQR Bounds')
plt.xlabel('Price (USD)')
plt.ylabel('Density')
plt.legend(fontsize=8, ncol=2)
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()


In [ ]:
import matplotlib.pyplot as plt

plt.figure(figsize=(12, 5))

# Build a list of price series in ticker order for clean axis labelling
price_data = [df[ticker].dropna().values for ticker in TICKERS]

plt.boxplot(price_data, labels=TICKERS, patch_artist=True,
            boxprops=dict(facecolor='lightsteelblue', color='steelblue'),
            medianprops=dict(color='darkblue', linewidth=1.5),
            whiskerprops=dict(color='steelblue'),
            capprops=dict(color='steelblue'),
            flierprops=dict(marker='o', color='steelblue', alpha=0.3, markersize=2))

plt.title('Distribution of Adjusted Close Prices by Ticker (2010 — Present)')
plt.xlabel('Ticker')
plt.ylabel('Price (USD)')
plt.grid(True, axis='y', alpha=0.3)
plt.tight_layout()
plt.show()


In [ ]:
import matplotlib.pyplot as plt

n = 252  # approximately one trading year

recent_data = df.tail(n)

recent_mean = recent_data.mean().sort_values(ascending=False)
print(f'Mean closing prices over the last {n} trading days:')
print(recent_mean.round(2))

pct_change = ((recent_data.iloc[-1] - recent_data.iloc[0]) / recent_data.iloc[0] * 100).sort_values(ascending=False)
print(f'\nPercentage price change over the last {n} trading days:')
print(pct_change.round(2))

best_performer  = pct_change.idxmax()
worst_performer = pct_change.idxmin()
print(f'\nBest performer : {best_performer} ({round(pct_change[best_performer], 2)}%)')
print(f'Worst performer: {worst_performer} ({round(pct_change[worst_performer], 2)}%)')

plt.figure(figsize=(13, 5))
for ticker in TICKERS:
    plt.plot(recent_data.index, recent_data[ticker], label=ticker, linewidth=1)

plt.title(f'Closing Prices Over the Last {n} Trading Days')
plt.xlabel('Date')
plt.ylabel('Price (USD)')
plt.legend(fontsize=8, ncol=2)
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()


In [ ]:
import pandas_datareader.data as web
import pandas as pd

cpi_raw = web.DataReader('CPIAUCSL', 'fred', start='2010-01-01')
cpi_raw.index = pd.to_datetime(cpi_raw.index)
cpi_raw.columns = ['CPI']

df_monthly = df.resample('ME').last()

df_monthly.index = df_monthly.index.to_period('M').to_timestamp()
cpi_raw.index    = cpi_raw.index.to_period('M').to_timestamp()

df_with_cpi = df_monthly.merge(cpi_raw, left_index=True, right_index=True, how='inner')

for ticker in TICKERS:
    df_with_cpi[ticker + '_Adj'] = df_with_cpi[ticker] * (100 / df_with_cpi['CPI'])

print('Rows after CPI merge:', len(df_with_cpi))
print(df_with_cpi[[TICKERS[0], TICKERS[0] + '_Adj', 'CPI']].head())


In [ ]:
from sklearn.neural_network import MLPRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error
import numpy as np
import matplotlib.pyplot as plt

ticker          = 'MSFT'
window_size     = 12   # number of past months used as input features
forecast_horizon = 24  # number of months ahead to evaluate

price_series = df_with_cpi[ticker + '_Adj'].dropna().values

X = []
y = []

for i in range(len(price_series) - window_size):
    X.append(price_series[i:i + window_size])
    y.append(price_series[i + window_size])

X = np.array(X)
y = np.array(y)

split = int(len(X) * 0.8)

X_train = X[:split]
X_test  = X[split:]
y_train = y[:split]
y_test  = y[split:]

model = MLPRegressor(
    hidden_layer_sizes=(50,),
    max_iter=500,
    random_state=42,
    early_stopping=True,
    validation_fraction=0.1,
    n_iter_no_change=10
)

model.fit(X_train, y_train)

plt.figure(figsize=(8, 4))
plt.plot(model.loss_curve_, label='Training Loss', color='steelblue')
plt.title(f'Training Loss over Iterations — {ticker}')
plt.xlabel('Iteration')
plt.ylabel('Loss')
plt.legend()
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

y_pred = model.predict(X_test)

y_test_horizon = y_test[:forecast_horizon]
y_pred_horizon = y_pred[:forecast_horizon]

mae  = mean_absolute_error(y_test_horizon, y_pred_horizon)
rmse = np.sqrt(mean_squared_error(y_test_horizon, y_pred_horizon))

print(f'Ticker           : {ticker}')
print(f'Forecast horizon : {forecast_horizon} months')
print(f'MAE              : ${mae:,.2f}')
print(f'RMSE             : ${rmse:,.2f}')

# Plot predicted vs actual
plt.figure(figsize=(10, 5))
plt.plot(y_test_horizon, label='Actual Price (Adj.)', color='steelblue')
plt.plot(y_pred_horizon, label='Predicted Price',     color='orange')
plt.title(f'Predicted vs Actual Inflation-Adjusted Prices — {ticker}')
plt.xlabel('Time (months)')
plt.ylabel('Adjusted Price (USD, base 1982–84)')
plt.legend()
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()


In [ ]:
from sklearn.neural_network import MLPRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error
import numpy as np
import matplotlib.pyplot as plt

def train_and_evaluate_mlp(price_series, window_size, forecast_horizon):
    X = []
    y = []

    for i in range(len(price_series) - window_size):
        X.append(price_series[i:i + window_size])
        y.append(price_series[i + window_size])

    X = np.array(X)
    y = np.array(y)

    split = int(len(X) * 0.8)

    X_train, X_test = X[:split], X[split:]
    y_train, y_test = y[:split], y[split:]

    mlp = MLPRegressor(
        hidden_layer_sizes=(50,),
        max_iter=500,
        random_state=42,
        early_stopping=True,
        validation_fraction=0.1,
        n_iter_no_change=10
    )
    mlp.fit(X_train, y_train)

    y_pred = mlp.predict(X_test)

    y_test_horizon = y_test[:forecast_horizon]
    y_pred_horizon = y_pred[:forecast_horizon]

    mae  = mean_absolute_error(y_test_horizon, y_pred_horizon)
    rmse = np.sqrt(mean_squared_error(y_test_horizon, y_pred_horizon))

    return mae, rmse, y_test_horizon, y_pred_horizon

# Sensitivity grid
ticker       = 'MSFT'
price_series = df_with_cpi[ticker + '_Adj'].dropna().values

window_sizes      = [6, 12, 24]
forecast_horizons = [24, 36, 48]

print(f'Sensitivity analysis for {ticker}')
print(f'{"Window":>8}  {"Horizon":>8}  {"MAE":>10}  {"RMSE":>10}')
print('-' * 44)

for window_size in window_sizes:
    for forecast_horizon in forecast_horizons:
        mae, rmse, _, _ = train_and_evaluate_mlp(price_series, window_size, forecast_horizon)
        print(f'{window_size:>8}  {forecast_horizon:>8}  {mae:>10.2f}  {rmse:>10.2f}')

fig, axes = plt.subplots(1, 3, figsize=(15, 4), sharey=True)

for ax, window_size in zip(axes, window_sizes):
    mae, rmse, y_test_h, y_pred_h = train_and_evaluate_mlp(price_series, window_size, 24)
    ax.plot(y_test_h, label='Actual',    color='steelblue')
    ax.plot(y_pred_h, label='Predicted', color='orange')
    ax.set_title(f'Window = {window_size} months\nMAE={mae:.1f}, RMSE={rmse:.1f}')
    ax.set_xlabel('Time (months)')
    ax.set_ylabel('Adjusted Price (USD)')
    ax.legend(fontsize=8)
    ax.grid(True, alpha=0.3)

plt.suptitle(f'Sensitivity to Window Size — {ticker} (Horizon = 24 months)', y=1.02)
plt.tight_layout()
plt.show()


In [ ]:
import matplotlib.pyplot as plt
import numpy as np

window_size      = 12
forecast_horizon = 24

rmse_by_ticker = {}
mae_by_ticker  = {}

for ticker in TICKERS:
    price_series = df_with_cpi[ticker + '_Adj'].dropna().values
    mae, rmse, _, _ = train_and_evaluate_mlp(price_series, window_size, forecast_horizon)
    rmse_by_ticker[ticker] = rmse
    mae_by_ticker[ticker]  = mae

sorted_tickers = sorted(rmse_by_ticker, key=rmse_by_ticker.get)
sorted_rmse    = [rmse_by_ticker[t] for t in sorted_tickers]

print(f'RMSE by ticker (window = {window_size}, horizon = {forecast_horizon} months):')
for ticker in sorted_tickers:
    print(f'  {ticker:>4}  MAE={mae_by_ticker[ticker]:>8.2f}  RMSE={rmse_by_ticker[ticker]:>8.2f}')

plt.figure(figsize=(10, 5))
bar_colours = ['steelblue' if t != sorted_tickers[0] else 'darkblue' for t in sorted_tickers]
plt.bar(sorted_tickers, sorted_rmse, color=bar_colours)
plt.title(f'Forecast RMSE by Ticker — Window={window_size} months, Horizon={forecast_horizon} months')
plt.xlabel('Ticker')
plt.ylabel('RMSE (Adjusted USD)')
plt.grid(True, axis='y', alpha=0.3)
plt.tight_layout()
plt.show()
